In [1]:
# --------------------------------------------
# Forecast & Inventory Risk Analytics
# Dataset Generator (Historical 2020-2023)
# Author: Diego Loera
# Purpose: Generate reproducible supply chain datasets
# Output: dim_product.csv, fact_sales.csv, fact_forecast.csv, fact_inventory.csv
# --------------------------------------------

# --- 1) Setup ---
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

OUT_DIR = Path("./ForecastProject_dataset")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- 2) Configuration ---
start_month = "2020-01-01"
end_month   = "2025-12-01"
months = pd.date_range(start=start_month, end=end_month, freq="MS")  # month start dates

n_products = 40

families = ["Electrical", "Fasteners", "Interior", "Trim"]
family_probs = [0.30, 0.40, 0.10, 0.20]  # you can bias if desired

# Unit costs per family (realistic-ish ranges in MXN)
cost_ranges = {
    "Electrical": (120, 420),
    "Fasteners": (10, 55),
    "Interior": (80, 260),
    "Trim": (60, 220),
}

# Baseline demand ranges per family (units/month)
demand_ranges = {
    "Electrical": (900, 4500),
    "Fasteners": (1500, 8000),
    "Interior": (600, 3000),
    "Trim": (700, 3500),
}

# Desired portfolio behavior
target_weighted_mape_range = (0.10, 0.25)   # 10–25%
target_avg_doi_range       = (30, 70)       # 30–70 days


# --- 3) Build dim_product ---
product_ids = [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)]
product_families = rng.choice(families, size=n_products, p=family_probs)

unit_costs = []
product_names = []
for pid, fam in zip(product_ids, product_families):
    lo, hi = cost_ranges[fam]
    unit_costs.append(np.round(rng.uniform(lo, hi), 2))
    product_names.append(f"Part_{pid}")

dim_product = pd.DataFrame({
    "product_id": product_ids,
    "product_name": product_names,
    "product_family": product_families,
    "unit_cost_mxn": unit_costs
})

dim_product.to_csv(OUT_DIR / "dim_product.csv", index=False)
dim_product.head()

,product_id,product_name,product_family,unit_cost_mxn
0,P001,Part_P001,Interior,158.69
1,P002,Part_P002,Fasteners,47.47
2,P003,Part_P003,Trim,172.04
3,P004,Part_P004,Fasteners,24.06
4,P005,Part_P005,Electrical,369.68


In [2]:
# --- 4) Demand generator helpers ---

def seasonal_factor(month_idx: int) -> float:
    """Mild seasonality across 12 months (0..11)."""
    # Sinusoidal seasonality in range ~[0.85, 1.15]
    return 1.0 + 0.15 * np.sin(2 * np.pi * (month_idx / 12.0))

def trend_factor(t: int, total_t: int) -> float:
    """Small upward trend over the entire horizon."""
    return 1.0 + 0.08 * (t / max(total_t - 1, 1))  # up to +8%

def sku_volatility(family: str) -> float:
    """Volatility differs by family."""
    if family == "Fasteners":
        return 0.20
    if family == "Electrical":
        return 0.18
    if family == "Trim":
        return 0.16
    return 0.14  # Interior

def sku_bias() -> float:
    """SKU forecast bias (systematic). Centered around 0, small magnitude."""
    return rng.normal(0.0, 0.04)  # ~ +/- 4% average bias

def clamp_int(x):
    return int(max(0, np.round(x)))


total_periods = len(months)

# Pre-assign per-SKU parameters (baseline demand, volatility, bias)
sku_params = {}
for pid, fam in zip(product_ids, product_families):
    d_lo, d_hi = demand_ranges[fam]
    base = rng.uniform(d_lo, d_hi)
    vol  = sku_volatility(fam)
    b    = sku_bias()
    sku_params[pid] = {"family": fam, "base_demand": base, "vol": vol, "bias": b}

sku_params[list(sku_params.keys())[0]]

{'family': np.str_('Interior'),
 'base_demand': 2194.35249678573,
 'vol': 0.14,
 'bias': -0.012373861588809536}

In [3]:
# --- 5) Generate actual sales (monthly) ---
rows_sales = []

for t, m in enumerate(months):
    m_idx = (m.month - 1)  # 0..11
    s = seasonal_factor(m_idx)
    tr = trend_factor(t, total_periods)

    for pid in product_ids:
        p = sku_params[pid]
        base = p["base_demand"]

        # Random noise ~ Normal around 1 with sku volatility
        noise = rng.normal(1.0, p["vol"])
        units = base * s * tr * noise

        rows_sales.append([m.date(), pid, clamp_int(units)])

fact_sales = pd.DataFrame(rows_sales, columns=["month", "product_id", "actual_units_sold"])
fact_sales.head()

,month,product_id,actual_units_sold
0,2020-01-01,P001,2496
1,2020-01-01,P002,6113
2,2020-01-01,P003,619
3,2020-01-01,P004,4999
4,2020-01-01,P005,1287


In [4]:
# --- 6) Generate forecast from actual with controlled error ---
# We'll create forecast by applying a controlled multiplicative error around 0
# Portfolio weighted MAPE will land in ~10-25% due to error amplitude below.

rows_forecast = []
# Error amplitude per SKU (some SKUs harder)
sku_difficulty = {pid: rng.uniform(0.08, 0.24) for pid in product_ids}  # 8–24%

for _, r in fact_sales.iterrows():
    pid = r["product_id"]
    actual = r["actual_units_sold"]
    p = sku_params[pid]

    # error term centered at (bias), with magnitude controlled by difficulty
    amp = sku_difficulty[pid]
    e = rng.normal(loc=p["bias"], scale=amp)  # multiplicative error component

    # forecast = actual * (1 + e) but ensure non-negative
    forecast = actual * (1.0 + e)
    rows_forecast.append([r["month"], pid, clamp_int(forecast)])

fact_forecast = pd.DataFrame(rows_forecast, columns=["month", "product_id", "forecast_units"])
fact_forecast.head()

,month,product_id,forecast_units
0,2020-01-01,P001,1945
1,2020-01-01,P002,4460
2,2020-01-01,P003,730
3,2020-01-01,P004,4741
4,2020-01-01,P005,1029


In [5]:
# --- 7) Generate inventory consistent with DOI target (30–70 days average) ---
# DOI approx = (ending_inventory_units / actual_units_sold) * 30
# So inv_units ≈ actual_units_sold * (DOI/30)

rows_inventory = []
# Base DOI per SKU (center around 45–55, with some dispersion)
sku_base_doi = {pid: rng.uniform(35, 65) for pid in product_ids}

for _, r in fact_sales.iterrows():
    pid = r["product_id"]
    actual = r["actual_units_sold"]

    # month-to-month DOI variation
    doi = max(5, rng.normal(sku_base_doi[pid], 12))  # allow variation but keep realistic

    inv_units = actual * (doi / 30.0)

    # Add extra "backlog/overproduction" occasionally to avoid overly smooth patterns
    if rng.random() < 0.08:
        inv_units *= rng.uniform(1.2, 1.8)

    rows_inventory.append([r["month"], pid, clamp_int(inv_units)])

fact_inventory = pd.DataFrame(rows_inventory, columns=["month", "product_id", "ending_inventory_units"])
fact_inventory.head()

,month,product_id,ending_inventory_units
0,2020-01-01,P001,3629
1,2020-01-01,P002,12792
2,2020-01-01,P003,1295
3,2020-01-01,P004,9708
4,2020-01-01,P005,1209


In [6]:
# --- 8) Validate portfolio metrics quickly (sanity checks) ---

# Join
df = fact_sales.merge(fact_forecast, on=["month", "product_id"], how="left") \
               .merge(fact_inventory, on=["month", "product_id"], how="left") \
               .merge(dim_product, on="product_id", how="left")

df["abs_error"] = (df["actual_units_sold"] - df["forecast_units"]).abs()
df["ape"] = np.where(df["actual_units_sold"] == 0, np.nan, df["abs_error"] / df["actual_units_sold"])

weighted_mape = df["abs_error"].sum() / max(df["actual_units_sold"].sum(), 1)
avg_doi = np.nanmean(np.where(df["actual_units_sold"] == 0, np.nan, (df["ending_inventory_units"] / df["actual_units_sold"]) * 30))

weighted_mape, avg_doi

(np.float64(0.12278303249816223), np.float64(49.46746137725917))

In [7]:
# --- 9) If needed, auto-tune to hit target ranges (simple loop) ---
# If your metrics are out of target, this will slightly adjust forecast difficulty & DOI.
# Usually SEED=42 already lands near target.

def regenerate_with_tuning(max_iter=10):
    global sku_difficulty, sku_base_doi, fact_forecast, fact_inventory

    for _ in range(max_iter):
        # Regenerate forecast
        rows_forecast = []
        for _, r in fact_sales.iterrows():
            pid = r["product_id"]
            actual = r["actual_units_sold"]
            p = sku_params[pid]
            amp = sku_difficulty[pid]
            e = rng.normal(loc=p["bias"], scale=amp)
            rows_forecast.append([r["month"], pid, clamp_int(actual * (1.0 + e))])
        fact_forecast = pd.DataFrame(rows_forecast, columns=["month", "product_id", "forecast_units"])

        # Regenerate inventory
        rows_inventory = []
        for _, r in fact_sales.iterrows():
            pid = r["product_id"]
            actual = r["actual_units_sold"]
            doi = max(5, rng.normal(sku_base_doi[pid], 12))
            inv_units = actual * (doi / 30.0)
            if rng.random() < 0.08:
                inv_units *= rng.uniform(1.2, 1.8)
            rows_inventory.append([r["month"], pid, clamp_int(inv_units)])
        fact_inventory = pd.DataFrame(rows_inventory, columns=["month", "product_id", "ending_inventory_units"])

        # Validate
        tmp = fact_sales.merge(fact_forecast, on=["month", "product_id"], how="left") \
                        .merge(fact_inventory, on=["month", "product_id"], how="left")
        tmp["abs_error"] = (tmp["actual_units_sold"] - tmp["forecast_units"]).abs()
        w_mape = tmp["abs_error"].sum() / max(tmp["actual_units_sold"].sum(), 1)
        tmp["doi"] = np.where(tmp["actual_units_sold"] == 0, np.nan, (tmp["ending_inventory_units"]/tmp["actual_units_sold"]) * 30)
        a_doi = np.nanmean(tmp["doi"])

        in_mape = target_weighted_mape_range[0] <= w_mape <= target_weighted_mape_range[1]
        in_doi  = target_avg_doi_range[0] <= a_doi <= target_avg_doi_range[1]

        if in_mape and in_doi:
            return w_mape, a_doi

        # Tune difficulty/doi targets slightly
        if w_mape < target_weighted_mape_range[0]:
            sku_difficulty = {k: min(v * 1.10, 0.30) for k, v in sku_difficulty.items()}
        elif w_mape > target_weighted_mape_range[1]:
            sku_difficulty = {k: max(v * 0.90, 0.05) for k, v in sku_difficulty.items()}

        if a_doi < target_avg_doi_range[0]:
            sku_base_doi = {k: min(v + 3, 90) for k, v in sku_base_doi.items()}
        elif a_doi > target_avg_doi_range[1]:
            sku_base_doi = {k: max(v - 3, 10) for k, v in sku_base_doi.items()}

    return w_mape, a_doi

w_mape2, a_doi2 = regenerate_with_tuning(max_iter=10)
w_mape2, a_doi2

(np.float64(0.12437468314107579), np.float64(50.01881968871802))

In [8]:
# --- 10) Export final CSVs ---
fact_sales.to_csv(OUT_DIR / "fact_sales.csv", index=False)
fact_forecast.to_csv(OUT_DIR / "fact_forecast.csv", index=False)
fact_inventory.to_csv(OUT_DIR / "fact_inventory.csv", index=False)

print("Saved files to:", OUT_DIR.resolve())
print(list(OUT_DIR.glob("*.csv")))

Saved files to: /content/ForecastProject_dataset
[PosixPath('ForecastProject_dataset/fact_forecast.csv'), PosixPath('ForecastProject_dataset/dim_product.csv'), PosixPath('ForecastProject_dataset/fact_inventory.csv'), PosixPath('ForecastProject_dataset/fact_sales.csv')]


In [9]:
#Create zip with all final files
import shutil
from google.colab import files

folder_to_zip = "output"

zip_filename = "project_dataset"
shutil.make_archive(zip_filename, 'zip', folder_to_zip)

print("ZIP created successfully!")

ZIP created successfully!


In [10]:
# Download dataset
files.download(f"{zip_filename}.zip")

print("Dataset downloaded successfully")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>